In [1]:
# ============================================
# 1. Environment Verification
# ============================================

import torch
import os
import pandas as pd

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

# ============================================
# 2. Verify Dataset Structure
# ============================================

DATA_ROOT = "/kaggle/input/crossmodal-misleading-video-dataset-v1"

for root, dirs, files in os.walk(DATA_ROOT):
    print(root)
    break
os.listdir(DATA_ROOT)

# ============================================
# 3. Load Train Annotations
# ============================================

train_ann_path = os.path.join(DATA_ROOT, "train/annotations/train_annotations.json")
train_df = pd.read_json(train_ann_path)

print("Train samples:", len(train_df))
train_df.head()

# =====================================================
# Test Annotation Integrity Check
# =====================================================

import json
import collections
import os

test_ann_path = os.path.join(DATA_ROOT, "test/annotations/test_annotations.json")

with open(test_ann_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Type:", type(test_data))
print("Total test samples:", len(test_data))

# Check video_id uniqueness
ids = [x["video_id"] for x in test_data]
print("Unique IDs:", len(set(ids)))
print("Duplicates:", len(ids) - len(set(ids)))

# Category distribution
print("\nCategory distribution:")
print(collections.Counter([x["category"] for x in test_data]))

# Subcategory distribution
print("\nSubcategory distribution:")
print(collections.Counter([x.get("subcategory") for x in test_data]))

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU Name: Tesla P100-PCIE-16GB
/kaggle/input/crossmodal-misleading-video-dataset-v1
Train samples: 1000
Type: <class 'list'>
Total test samples: 500
Unique IDs: 500
Duplicates: 0

Category distribution:
Counter({'misleading': 400, 'safe': 100})

Subcategory distribution:
Counter({'identity_fabrication': 100, 'perception_manipulation': 100, 'scientifically_unrealistic_scene': 100, 'surreal_content': 100, None: 100})


In [2]:
# =====================================================
# Cross-split Leakage Check (Corrected)
# =====================================================

train_ids = set(train_df["video_id"].tolist())
test_ids = set([x["video_id"] for x in test_data])

overlap = train_ids.intersection(test_ids)

print("Overlap count:", len(overlap))
print("Overlapping IDs:", overlap)

Overlap count: 0
Overlapping IDs: set()


In [3]:
# =====================================================
# 5. Create Unified Text Input (Fixed Version)
# =====================================================

def combine_text(row):
    parts = []
    
    # Visual description
    if isinstance(row["visual_description"], str):
        parts.append(row["visual_description"])
        
    # Audio transcript
    if isinstance(row["audio_transcript"], str):
        parts.append(row["audio_transcript"])
        
    # OCR text (list → join)
    if isinstance(row["ocr_text"], list):
        parts.append(" ".join(row["ocr_text"]))
    elif isinstance(row["ocr_text"], str):
        parts.append(row["ocr_text"])
        
    # Scene summary
    if isinstance(row["scene_summary"], str):
        parts.append(row["scene_summary"])
    
    return " ".join(parts)

train_df["combined_text"] = train_df.apply(combine_text, axis=1)

print("Example combined text:")
print(train_df["combined_text"].iloc[0][:500])

Example combined text:
An elderly man with gray hair is seated on an ornate white carved chair, speaking directly toward the camera. The background features wooden interior decor. The scene remains static, and subtitle-style text appears at the bottom of the screen. I've decided to create a new application. Crazy time. By investing just 5 taka, you can win 50,000 or even 100,000 taka on your first day. due to the current crisis in the country I've decided to create a new application crazy time by investing just 5 taka


In [4]:
# =====================================================
# 6. Install & Import Transformers
# =====================================================

!pip install -q transformers

from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch

In [5]:
# =====================================================
# 7. Load Pretrained BERT Tokenizer
# =====================================================

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

sample_text = train_df["combined_text"].iloc[0]

tokens = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

print("Input IDs shape:", tokens["input_ids"].shape)
print("Attention mask shape:", tokens["attention_mask"].shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Input IDs shape: torch.Size([1, 256])
Attention mask shape: torch.Size([1, 256])
